In [ ]:
from dotenv import load_dotenv
from src.datasources import glofas
from src.utils import blob

load_dotenv()

In [ ]:
STATION_NAME = "makurdi"
PRODUCT_TYPE = "ensemble"  # or "control"

## Check raw blobs

In [ ]:
raw_prefix = (
    f"ds-aa-nga-flooding/raw/glofas/reforecast/"
    f"glofas_raw_reforecast_{STATION_NAME}_{PRODUCT_TYPE}_"
)
raw_blobs = sorted(
    x for x in blob.list_container_blobs(name_starts_with=raw_prefix)
    if x.endswith(".grib")
)
print(f"Found {len(raw_blobs)} raw GRIB files")
for b in raw_blobs:
    print(" ", b)

## Process and upload

Reads each GRIB, extracts `dis24` at the station grid point across all leadtimes and ensemble members, concatenates, and uploads to:
`ds-aa-nga-flooding/processed/glofas/glofas_reforecast_{station}_{product_type}.parquet`

In [ ]:
glofas.process_glofas_reforecast(STATION_NAME, PRODUCT_TYPE)

## Verify output

In [ ]:
df = glofas.load_glofas_reforecast(STATION_NAME, PRODUCT_TYPE)
print(f"Rows:        {len(df):,}")
print(f"Columns:     {list(df.columns)}")
print(f"Leadtimes:   {sorted(df['leadtime'].unique())}")
if 'number' in df.columns:
    print(f"Ens members: {sorted(df['number'].unique())}")
print(f"Time range:  {df['time'].min().date()} → {df['time'].max().date()}")
df.head()